# Practical Assignment 10: Autoencoder for Anomaly Detection on Credit Card Dataset

**Aim:** Use an Autoencoder to implement anomaly detection on the credit card transaction dataset.

**Reference:** BE IT Course: Lab Practice IV Lab Manual Assignment No. 4 (Pages 32-39).

**Anomaly Detection Logic:**
1. Train the autoencoder **ONLY on Normal transactions** (`Class == 0`).
2. The network learns normal spending patterns with low reconstruction error.
3. When an anomalous **Fraud transaction** (`Class == 1`) is presented, the reconstruction error is significantly higher.
4. A threshold on the reconstruction error flags fraudulent transactions.


In [1]:
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'
import pandas as pd
import numpy as np
import tensorflow as tf
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

# Step 1: Load Credit Card Dataset and Separate Normal vs Fraud
csv_path = r'd:/workspace01/4th year/DL/LabPracticesIV_DEEP_LEARNING/dataset/Credit Card Dataset/creditcard.csv'
df = pd.read_csv(csv_path, nrows=15000)

# Standardize Amount and Time as done in Lab Manual Page 35
scaler = StandardScaler()
df['Amount'] = scaler.fit_transform(df[['Amount']])
df = df.drop(['Time'], axis=1)

normal_df = df[df['Class'] == 0].drop(['Class'], axis=1)
fraud_df = df[df['Class'] == 1].drop(['Class'], axis=1)

print(f'Normal records: {len(normal_df)}, Fraud records: {len(fraud_df)}')

X_train_normal, X_test_normal = train_test_split(normal_df.values, test_size=0.2, random_state=42)
input_dim = X_train_normal.shape[1]  # 29 features

# Step 2: Build Autoencoder Architecture (from Lab Manual Page 36)
autoencoder = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(input_dim,)),
    tf.keras.layers.Dense(14, activation='relu'),
    tf.keras.layers.Dense(7, activation='relu'),   # Bottleneck
    tf.keras.layers.Dense(14, activation='relu'),
    tf.keras.layers.Dense(input_dim, activation='linear')
])
autoencoder.summary()

# Step 3: Train Autoencoder ONLY on Normal Data with MSE Loss
autoencoder.compile(optimizer='adam', loss='mse')
history = autoencoder.fit(X_train_normal, X_train_normal, epochs=5, batch_size=64, validation_split=0.1, verbose=1)

# Step 4: Detect Anomalies based on Reconstruction Error
pred_normal = autoencoder.predict(X_test_normal, verbose=0)
mse_normal = np.mean((X_test_normal - pred_normal) ** 2, axis=1)

pred_fraud = autoencoder.predict(fraud_df.values, verbose=0)
mse_fraud = np.mean((fraud_df.values - pred_fraud) ** 2, axis=1)

print(f'\nNormal Transactions Mean MSE: {np.mean(mse_normal):.4f}')
print(f'Fraud (Anomaly) Transactions Mean MSE: {np.mean(mse_fraud):.4f}')

# Set Threshold at 95th percentile of normal reconstruction error
threshold = np.percentile(mse_normal, 95)
print(f'Anomaly Threshold: {threshold:.4f}')
fraud_detected = np.sum(mse_fraud > threshold)
print(f'Fraud Detection Recall: {(fraud_detected / len(mse_fraud))*100:.1f}% ({fraud_detected}/{len(mse_fraud)} detected)')


Normal records: 14939, Fraud records: 61
Model: "sequential_7"
┌─────────────────────────────────┬────────────────────────┬───────────────┐
│ Layer (type)                    │ Output Shape           │       Param # │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_18 (Dense)                │ (None, 14)             │           420 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_19 (Dense)                │ (None, 7)              │           105 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_20 (Dense)                │ (None, 14)             │           112 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_21 (Dense)                │ (None, 29)             │           435 │
└─────────────────────────────────┴────────────────────────┴───────────────┘
 Total params: 1,072 (4.19 KB)
 Trainable params: 1,072 (4.19 KB)
 Non-trainable params: 0